# YIFE End-to-End Demo
**Predicting Early-Stage Startup Success: YC-Inspired Feature Engineering**

This notebook walks through the complete YIFE pipeline in ~5 minutes:
1. Load the sample YC dataset
2. Explore YIFE features
3. Train XGBoost champion model
4. Evaluate with ROC curve + confusion matrix
5. Interpret with SHAP
6. Predict on new companies

> Paper: Gupta et al. (2026), PTESM 2026
> Repo: https://github.com/guptasiddharth2409/yife-startup-prediction


In [ ]:
# Install dependencies (Colab / fresh environment)
%pip install -q xgboost shap matplotlib seaborn scikit-learn pandas numpy pyarrow


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

# Load sample dataset
df = pd.read_csv('../data/raw/sample_yc_data.csv')
print(f'Shape: {df.shape}')
print(f'Success rate: {df.success.mean():.1%}')
df.head(10)


In [ ]:
# Exploratory analysis: success rate by industry
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Success by industry
industry_sr = df.groupby('industry_category')['success'].mean().sort_values(ascending=False)
industry_sr.plot(kind='bar', ax=axes[0], color='#2563eb', edgecolor='white')
axes[0].set_title('Success Rate by Industry', fontweight='bold')
axes[0].set_ylabel('Success Rate'); axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Funding distribution by success
df.groupby('success')['total_funding_usd'].apply(np.log1p).unstack().T.plot(
    kind='box', ax=axes[1], color={'boxes':'#2563eb','medians':'#dc2626','whiskers':'#6b7280','caps':'#6b7280'})
axes[1].set_title('Log Funding by Success Label', fontweight='bold')
axes[1].set_xticklabels(['Not Successful', 'Successful'])

plt.tight_layout(); plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Feature engineering on sample data
df_enc = df.copy()
for col in ['industry_category', 'location']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

FEATURES = ['total_funding_usd', 'num_funding_rounds', 'team_size',
            'faang_experience', 'elite_edu', 'ai_flag', 'tier1_vc_investor',
            'industry_category', 'location']

X = df_enc[FEATURES].astype(float)
y = df_enc['success']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# Train XGBoost champion
model = XGBClassifier(n_estimators=300, learning_rate=0.05,
                      eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Not Successful','Successful']))
print(f'AUROC: {roc_auc_score(y_test, y_prob):.4f}')


In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_val = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(5, 4), dpi=150)
ax.plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'XGBoost (AUC = {auc_val:.3f})')
ax.plot([0,1],[0,1],'k:',lw=1.2, label='Random Chance')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — XGBoost on Sample Data', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
import shap

# SHAP Global Feature Importance
explainer  = shap.TreeExplainer(model)
shap_vals  = explainer.shap_values(X_test)
mean_abs   = np.abs(shap_vals).mean(axis=0)

feat_imp = pd.Series(mean_abs, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(6, 3.5), dpi=150)
colors = ['#1d4ed8' if v >= 0.12 else '#3b82f6' if v >= 0.07 else '#93c5fd' for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=colors[::-1][:len(feat_imp)], edgecolor='white')
ax.set_xlabel('Mean |SHAP| Value'); ax.grid(axis='x', alpha=0.3)
ax.set_title('SHAP Feature Importance — XGBoost (YIFE)', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 3 most important features:')
print(feat_imp.sort_values(ascending=False).head(3).to_string())


In [ ]:
# Predict on hypothetical new companies
new_companies = pd.DataFrame({
    'company': ['NewStartup_A', 'NewStartup_B', 'NewStartup_C'],
    'total_funding_usd':  [5_000_000, 500_000, 50_000_000],
    'num_funding_rounds': [3, 1, 5],
    'team_size':          [3, 1, 4],
    'faang_experience':   [1, 0, 1],
    'elite_edu':          [1, 0, 1],
    'ai_flag':            [1, 0, 1],
    'tier1_vc_investor':  [1, 0, 1],
    'industry_category':  [2, 0, 1],  # encoded
    'location':           [3, 1, 3],  # encoded
})

X_new    = new_companies[FEATURES].astype(float)
probs    = model.predict_proba(X_new)[:, 1]
new_companies['success_probability'] = probs.round(3)
new_companies['prediction'] = np.where(probs >= 0.5, 'SUCCESS', 'NOT SUCCESSFUL')

print('=== Predictions for New Companies ===')
print(new_companies[['company','success_probability','prediction']].to_string(index=False))
